# CineEmbed — 04 Train DEC (MVP scope)

Trains DEC for the **intermediate report MVP**:
- Run: `dec_z64_k21` (DEC at z=64, k=21 — matches primary_genre class count)

**Prerequisite**: `02_train_ae.ipynb` must have completed; `ae_z64.pt` must exist in `artifacts/models/`.
**Compute estimate**: ~10 min on Colab T4.
**Deferred for final report**: 8 more DEC runs covering z={32, 128} × k={10, 21, 30} and z=64 × k={10, 30}.

In [ ]:
import os, sys, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Auth: try GitHub token from Colab Secrets (for private repos), fall back to public URL
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    except Exception:
        REPO_URL = "https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')

    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        # Pull latest if you've pushed updates from local
        get_ipython().system(f'cd {REPO_ROOT} && git pull -q')

    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import torch

from cineembed import data, backbone, heads, losses, train, eval as cev

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Repo: {REPO_ROOT}\nArtifacts: {ARTIFACTS}\nDevice: {DEVICE}")

In [ ]:
X, feature_names = data.load_feature_matrix(ARTIFACTS / 'feature_matrix.npz')
labels = data.get_labels(ARTIFACTS / 'movies_eda_final.csv')
block_slices = data.get_block_indices(feature_names)
has_bio = X[:, block_slices['director'].start + 64].clone()

# Verify AE prerequisite (only ae_z64 required for this MVP run)
p = ARTIFACTS / 'models' / 'ae_z64.pt'
assert p.exists(), f"Missing prerequisite: {p} — run 02_train_ae.ipynb first"
print("AE prerequisite OK; X:", X.shape)

In [ ]:
BLOCK_DIMS = {b: (slc.stop - slc.start) for b, slc in block_slices.items()}
PROJ_DIMS = backbone.DEFAULT_PROJ_DIMS
w_blocks = losses.compute_block_weights(X.numpy(), block_slices, w_min=0.1, w_max=10.0)
train_idx, val_idx = data.train_val_split(X.shape[0], val_frac=0.1, seed=42)
train_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=True,
                                     indices=train_idx, block_slices=block_slices, seed=42)
val_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=False,
                                    indices=val_idx, block_slices=block_slices, seed=42)

In [ ]:
def _compute_full_latents_and_counts(model, X, has_bio, block_slices, device):
    """Forward pass over the entire dataset; return (z_array, per-cluster counts)."""
    model.eval()
    full_loader = data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                         block_slices=block_slices)
    z_chunks, q_argmax_chunks = [], []
    with torch.no_grad():
        for batch in full_loader:
            blk = {k_: v.to(device) for k_, v in batch['blocks'].items()}
            z, _, q = model(blk)
            z_chunks.append(z.cpu().numpy())
            q_argmax_chunks.append(q.argmax(dim=1).cpu().numpy())
    z_all = np.concatenate(z_chunks, axis=0)
    assignments = np.concatenate(q_argmax_chunks, axis=0)
    counts = np.bincount(assignments, minlength=model.n_clusters)
    return z_all, torch.from_numpy(counts)


def train_dec_run(run_name, z_dim, k, ae_checkpoint_path, n_epochs=50,
                   cluster_size_floor=0.001):
    """DEC training with explicit cluster-collapse re-init between epochs."""
    torch.manual_seed(42)
    import time as _time
    _t0 = _time.time()
    print(f"\n{'='*72}")
    print(f"[{_time.strftime('%H:%M:%S')}] BEGIN run='{run_name}' z={z_dim} k={k}")
    print(f"{'='*72}")

    bb = backbone.MultiModalBackbone(BLOCK_DIMS, PROJ_DIMS, hidden_dim=128, latent_dim=z_dim)
    ae_head = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
    train.load_checkpoint(ae_head, ae_checkpoint_path, device=DEVICE)
    ae_head = ae_head.to(DEVICE)

    # Compute initial latents on full data for k-means init
    ae_head.eval()
    z_init = []
    full_loader = data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                         block_slices=block_slices)
    with torch.no_grad():
        for batch in full_loader:
            blk = {k_: v.to(DEVICE) for k_, v in batch['blocks'].items()}
            z_init.append(ae_head.encode(blk).cpu().numpy())
    z_init = np.concatenate(z_init, axis=0)

    dec_head = heads.DECHead(bb, ae_head.decoder, n_clusters=k, latent_dim=z_dim)
    dec_head.initialize_centers(z_init, seed=42)
    dec_head = dec_head.to(DEVICE)

    # Manual training loop because we need a per-epoch cluster-count check.
    opt = torch.optim.Adam(dec_head.parameters(), lr=1e-3, weight_decay=1e-5)
    history = {'train_loss': [], 'val_loss': [], 'n_reinit': []}
    best_val = float('inf'); patience = 0
    ckpt_path = ARTIFACTS / 'models' / f'{run_name}.pt'

    n_total = X.shape[0]

    for epoch in range(n_epochs):
        dec_head.train()
        train_losses = []
        for batch in train_loader:
            blk = {k_: v.to(DEVICE) for k_, v in batch['blocks'].items()}
            hb = batch['has_bio'].to(DEVICE)
            opt.zero_grad()
            z, decoded, _ = dec_head(blk)
            loss, _, _ = losses.dec_loss(z, decoded, blk, dec_head.cluster_centers,
                                           hb, w_blocks, lambda_recon=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(dec_head.parameters(), 1.0)
            opt.step()
            train_losses.append(float(loss.item()))
        history['train_loss'].append(sum(train_losses) / max(len(train_losses), 1))

        # Cluster-collapse re-init (spec §10 mitigation)
        z_full, cluster_counts = _compute_full_latents_and_counts(
            dec_head, X, has_bio, block_slices, DEVICE,
        )
        n_reinit = dec_head.reinit_collapsed_centers(
            cluster_counts=cluster_counts, z_pool=z_full,
            n_total=n_total, size_floor=cluster_size_floor, seed=42 + epoch,
        )
        history['n_reinit'].append(n_reinit)
        if n_reinit > 0:
            print(f"  [epoch {epoch}] re-initialized {n_reinit} collapsed clusters")

        # Validation pass
        dec_head.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                blk = {k_: v.to(DEVICE) for k_, v in batch['blocks'].items()}
                hb = batch['has_bio'].to(DEVICE)
                z, decoded, _ = dec_head(blk)
                v_loss, _, _ = losses.dec_loss(z, decoded, blk, dec_head.cluster_centers,
                                                 hb, w_blocks, lambda_recon=0.1)
                val_losses.append(float(v_loss.item()))
        val_avg = sum(val_losses) / max(len(val_losses), 1)
        history['val_loss'].append(val_avg)

        _marker = '↓' if (best_val - val_avg) > 1e-4 else '·'
        print(f"[{_time.strftime('%H:%M:%S')}] epoch {epoch+1:3d}/{n_epochs} | "
              f"train={history['train_loss'][-1]:.4f} val={val_avg:.4f} {_marker} "
              f"(best={best_val:.4f}, reinit={n_reinit})")

        if best_val - val_avg > 1e-4:
            best_val = val_avg; patience = 0
            ckpt_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({'model_state': dec_head.state_dict(), 'epoch': epoch,
                        'val_loss': val_avg, 'history': history}, ckpt_path)
        else:
            patience += 1
            if patience >= 8:
                break

    # Final cluster assignments via DEC argmax
    train.load_checkpoint(dec_head, ckpt_path, device=DEVICE)
    dec_head.eval()
    full_batches = list(data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                                block_slices=block_slices))
    c_ids = cev.cluster_assignments_dec(dec_head, full_batches, device=DEVICE)
    metrics = cev.evaluate_run(c_ids, labels)
    metrics['run_name'] = run_name
    metrics['z_dim'] = z_dim
    metrics['k'] = k
    metrics['total_reinit'] = int(sum(history['n_reinit']))
    metrics['final_val_loss'] = best_val
    _elapsed = _time.time() - _t0
    print(f"[{_time.strftime('%H:%M:%S')}] END   run='{run_name}' in {_elapsed/60:.1f} min | "
          f"genre_NMI={metrics.get('genre_nmi', 0):.3f} | "
          f"total_reinit={metrics.get('total_reinit', 0)}")
    return metrics


In [ ]:
all_metrics = []

# Run dec_z64_k21 (z=32 and z=128 latent dims deferred to final report)
ae_ckpt = ARTIFACTS / 'models' / 'ae_z64.pt'
m = train_dec_run('dec_z64_k21', z_dim=64, k=21, ae_checkpoint_path=ae_ckpt)
all_metrics.append(m)
print(f"[dec_z64_k21] genre_NMI={m['genre_nmi']:.3f}  decade_NMI={m['decade_nmi']:.3f}  lang_NMI={m['lang_nmi']:.3f}")

# Append to results.json
results_path = ARTIFACTS / 'eval' / 'results.json'
existing = json.loads(results_path.read_text()) if results_path.exists() else {}
for m in all_metrics:
    existing[m['run_name']] = m
results_path.write_text(json.dumps(existing, indent=2))

## DEC runs deferred to final report

This MVP notebook trained 1 of 9 planned DEC runs. After intermediate, uncomment the cell below to add the remaining 8:

In [ ]:
# ─── Deferred for final report — uncomment to run after intermediate ─────────
# all_metrics = []
# for z in [32, 64, 128]:
#     for k in [10, 21, 30]:
#         if z == 64 and k == 21:
#             continue   # already trained in MVP
#         ae_ckpt = ARTIFACTS / 'models' / f'ae_z{z}.pt'
#         m = train_dec_run(f'dec_z{z}_k{k}', z_dim=z, k=k, ae_checkpoint_path=ae_ckpt)
#         all_metrics.append(m)
#         print(f"[dec_z{z}_k{k}] genre_NMI={m['genre_nmi']:.3f}  decade_NMI={m['decade_nmi']:.3f}")
#
# results_path = ARTIFACTS / 'eval' / 'results.json'
# existing = json.loads(results_path.read_text()) if results_path.exists() else {}
# for m in all_metrics:
#     existing[m['run_name']] = m
# results_path.write_text(json.dumps(existing, indent=2))